In [3]:
# Find K nearest neighbors for each case, excluding re-hearings of the same case

import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from matplotlib import pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics.pairwise import cosine_distances

from utils import parse_casenum

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

K = 30  # number of nearest neighbors to compare to

In [4]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data.parquet"))
df_emb = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "embeddings.parquet"))

In [5]:
# merge on case nums to the embeddings data set

df_emb = df_emb.merge(
    df[['date', 'year', 'item_no', 'title']], 
    on=['date', 'year', 'item_no'], 
    how='left'
)
df_emb['canonical_casenum'] = df_emb['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])



In [6]:
# dict of excluded neighbors, mapping case index num to set of indices that are re-hearings of same case

excluded_neighbors = {}
for i, row in df_emb.iterrows():
    canonical_casenum = row['canonical_casenum']
    if canonical_casenum not in excluded_neighbors:
        excluded_neighbors[canonical_casenum] = set()
    excluded_neighbors[canonical_casenum].add(i)



In [7]:
# turn embeddings into np array

embeddings = np.array(df_emb['embedding'].tolist())
n = len(embeddings)

In [8]:
# cosine distances between all embedding points

dist_matrix = cosine_distances(embeddings)

In [9]:
# step 1: find each cases's K nearest neighbors and their distances (excluding re-hearings of same case)

knn_indices = {}
k_distances = np.zeros(n)

for i, row in df_emb.iterrows():
    dists = dist_matrix[i].copy()
    dists[i] = np.inf # exclude self
    for j in excluded_neighbors.get(i, set()):
        dists[j] = np.inf # exclude re-hearings of same case
    neighbors = np.argsort(dists)[:K]
    knn_indices[i] = neighbors
    k_distances[i] = dists[neighbors[-1]] # distance to Kth nearest neighbor
    

In [10]:
# step 2: local reachability density for each case

lrd = np.zeros(n)

for i, row in df_emb.iterrows():
    reach_dists = []
    for j in knn_indices[i]:
        reach_dist = max(dist_matrix[i, j], k_distances[j])
        reach_dists.append(reach_dist)
    lrd[i] = 1.0 / np.mean(reach_dists) 



In [11]:
# step 3: lof score

lof_scores = np.zeros(n)

for i, row in df_emb.iterrows():
    ratios = []
    for j in knn_indices[i]:
        ratios.append(lrd[j] / lrd[i])
    lof_scores[i] = np.mean(ratios)

df_emb['lof_score'] = lof_scores

In [12]:
# step 4: simple avg cosine distance

avg_knn_dist = np.zeros(n)

for i, row in df_emb.iterrows():
    knn_dists = []
    for j in knn_indices[i]:
        knn_dists.append(dist_matrix[i, j])
    avg_knn_dist[i] = np.mean(knn_dists)

df_emb['avg_knn_dist'] = avg_knn_dist


In [13]:
df = df.merge(
    df_emb[['date', 'year', 'item_no', 'lof_score', 'avg_knn_dist']], 
    on=['date', 'year', 'item_no'], 
    how='left'
)

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_knn_dists.parquet"))
